<a href="https://colab.research.google.com/github/ketanasindhusomisetty/GENAI_2520030487/blob/main/SKILL/GenAI_Exp11_Sampling_Sweep_Failures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
# Experiment 11: Sampling sweep over temperature, top-k and top-p on the golden set; classify failures
!pip -q install pymongo
import time, random
import pandas as pd
from google.colab import userdata
from google import genai
from google.genai import types
from pymongo import MongoClient


MODEL = "gemini-3.1-flash-lite"

# Initialize Gemini client
client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY"),
    http_options={"timeout": 30000}
)

# Get MongoDB URI securely from Colab Secrets
from urllib.parse import quote_plus

raw_uri = userdata.get("MONGODB_URI")

# Encode special characters in the password
scheme, remainder = raw_uri.split("://", 1)
credentials, host = remainder.rsplit("@", 1)
username, password = credentials.split(":", 1)

encoded_password = quote_plus(password)

mongodb_uri = f"{scheme}://{username}:{encoded_password}@{host}"

# Connect to MongoDB
mongo_client = MongoClient(
    mongodb_uri,
    serverSelectionTimeoutMS=8000
)

db = mongo_client["genai_course"]

# Load golden dataset
golden = list(db.golden_set.find({}, {"_id": 0}))

random.seed(7)
subset = random.sample(golden, 8)

print("Loaded golden set:", len(golden), "| sweeping on", len(subset), "questions")

random.seed(7); subset = random.sample(golden, 8)          # 8 questions x 5 configs = 40 calls (free-tier friendly)
print("Loaded golden set:", len(golden), "| sweeping on", len(subset), "questions")

CONFIGS = {
    "greedy T=0":            dict(temperature=0.0),
    "T=0.7, top_p=0.9":      dict(temperature=0.7, top_p=0.9),
    "T=1.0, top_k=40":       dict(temperature=1.0, top_k=40),
    "T=1.5, top_p=1.0":      dict(temperature=1.5, top_p=1.0),
    "T=2.0, top_k=100":      dict(temperature=2.0, top_k=100),
}

def ask(prompt, cfg, retries=4):
    for a in range(retries):
        try:
            return client.models.generate_content(model=MODEL, contents=prompt,
                       config=types.GenerateContentConfig(max_output_tokens=200, **cfg))
        except Exception as e:
            print("retry", a + 1, str(e)[:50]); time.sleep(5 * (a + 1))
    return None

def classify(ans, keywords, out_tokens):
    a = (ans or "").lower().replace("-", " ")
    if not a.strip():                                             return "empty"
    if all(k.lower() in a for k in keywords):                    return "correct"
    if any(s in a for s in ["i don't", "i do not", "cannot", "please provide", "not sure"]): return "refusal / unknown"
    if out_tokens and out_tokens >= 190:                          return "verbose / truncated"
    return "incorrect / hallucinated"

rows = []
for name, cfg in CONFIGS.items():
    for g in subset:
        t = time.time()
        r = ask("Answer in one or two sentences: " + g["question"], cfg)
        ans = (r.text if r and r.text else "")
        out_t = r.usage_metadata.candidates_token_count if r else 0
        rows.append({"config": name, **cfg, "id": g["id"], "label": classify(ans, g["expected_keywords"], out_t),
                     "latency_s": round(time.time() - t, 2), "output_tokens": out_t, "answer": ans[:70].replace("\n", " ")})
        time.sleep(4)

df = pd.DataFrame(rows)
db.sampling_sweep.delete_many({}); db.sampling_sweep.insert_many(df.drop(columns=["answer"]).to_dict("records"))
summary = df.groupby("config").agg(accuracy=("label", lambda s: round((s == "correct").mean(), 2)),
                                   avg_latency_s=("latency_s", "mean"), avg_output_tokens=("output_tokens", "mean")).round(2)
print("\nSweep summary:"); print(summary.sort_values("accuracy", ascending=False).to_string())
print("\nFailure classification sheet (counts per config):")
print(pd.crosstab(df.config, df.label).to_string())
print("\nExamples of failures:")
print(df[df.label != "correct"][["config", "id", "label", "answer"]].head(10).to_string(index=False))

Loaded golden set: 32 | sweeping on 8 questions
Loaded golden set: 32 | sweeping on 8 questions
retry 1 504 DEADLINE_EXCEEDED. {'error': {'code': 504, 'me

Sweep summary:
                  accuracy  avg_latency_s  avg_output_tokens
config                                                      
T=1.5, top_p=1.0      1.00           6.19              39.75
T=2.0, top_k=100      1.00           4.26              41.00
T=0.7, top_p=0.9      0.88           5.10              36.88
T=1.0, top_k=40       0.88           1.22              40.88
greedy T=0            0.88           6.65              37.38

Failure classification sheet (counts per config):
label             correct  incorrect / hallucinated
config                                             
T=0.7, top_p=0.9        7                         1
T=1.0, top_k=40         7                         1
T=1.5, top_p=1.0        8                         0
T=2.0, top_k=100        8                         0
greedy T=0              7              

In [14]:
print("Total golden questions:", db.golden_set.count_documents({}))

Total golden questions: 0


In [15]:

# Inspect MongoDB database and collections

print("Connected database:", db.name)

print("\nAvailable collections:")
print(db.list_collection_names())

print("\nDocument counts:")

for collection_name in db.list_collection_names():
    count = db[collection_name].count_documents({})
    print(collection_name, ":", count)


Connected database: genai_course

Available collections:
[]

Document counts:


In [16]:

print("Available MongoDB databases:")

databases = mongo_client.list_database_names()

for database_name in databases:
    print("\nDatabase:", database_name)

    current_db = mongo_client[database_name]
    collections = current_db.list_collection_names()

    print("Collections:", collections)

    for collection_name in collections:
        count = current_db[collection_name].count_documents({})
        print(f"  {collection_name}: {count} documents")


Available MongoDB databases:

Database: bankdb
Collections: ['transactions', 'customers', '1. customers  2. transactions']
  transactions: 15 documents
  customers: 9 documents
  1. customers  2. transactions: 0 documents

Database: sample_mflix
Collections: ['users', 'comments', 'embedded_movies', 'sessions', 'theaters', 'movies']
  users: 185 documents
  comments: 41079 documents
  embedded_movies: 3483 documents
  sessions: 1 documents
  theaters: 1564 documents
  movies: 21349 documents

Database: admin
Collections: []

Database: local
Collections: ['oplog.rs']
  oplog.rs: 18 documents


In [17]:
from urllib.parse import urlparse
from google.colab import userdata

uri = userdata.get("MONGODB_URI")
print("Connected Atlas host:", urlparse(uri).hostname)

Connected Atlas host: cluster0.upbiakk.mongodb.net


In [18]:

import json

# Reuse the MongoDB connection that we already tested
db = mongo_client["genai_course"]

# Original golden dataset from Experiment 06
GOLDEN = [
    ("What does BPE stand for?", ["byte", "pair"], "Module I"),
    ("Which tokenizer does BERT use?", ["wordpiece"], "Module I"),
    ("Which tokenizer family do T5 and mT5 use?", ["sentencepiece"], "Module I"),
    ("Why does Telugu text usually cost more tokens than English?", ["token"], "Module I"),
    ("What is a knowledge cutoff of an LLM?", ["date"], "Module I"),
    ("What is a hallucination in an LLM?", ["false"], "Module I"),
    ("What is next-token prediction?", ["next", "token"], "Module I"),
    ("What is a context window?", ["token"], "Module I"),
    ("Name the three vectors used in attention.", ["query", "key", "value"], "Module II"),
    ("Which function turns attention scores into weights?", ["softmax"], "Module II"),
    ("Is GPT encoder-only or decoder-only?", ["decoder"], "Module II"),
    ("Is BERT encoder-only or decoder-only?", ["encoder"], "Module II"),
    ("What kind of architecture is T5?", ["encoder", "decoder"], "Module II"),
    ("What does a higher temperature do during sampling?", ["random"], "Module II"),
    ("What is top-k sampling?", ["k"], "Module II"),
    ("What is top-p sampling also called?", ["nucleus"], "Module II"),
    ("What does TF-IDF stand for?", ["term frequency", "inverse document frequency"], "Module III"),
    ("Name a classic sparse ranking function used in search engines.", ["bm25"], "Module III"),
    ("What does HNSW stand for?", ["hierarchical", "navigable", "small world"], "Module III"),
    ("What does RAG stand for?", ["retrieval", "augmented", "generation"], "Module III"),
    ("What does MRR stand for in retrieval evaluation?", ["mean", "reciprocal", "rank"], "Module III"),
    ("Why do we add overlap when chunking documents?", ["context"], "Module III"),
    ("What is few-shot prompting?", ["example"], "Module IV"),
    ("Which Python library is used for schema-validated structured outputs in this course?", ["pydantic"], "Module IV"),
    ("What is prompt injection?", ["instruction"], "Module IV"),
    ("What does LoRA stand for?", ["low", "rank", "adaptation"], "Module V"),
    ("What does the Q in QLoRA refer to?", ["quantiz"], "Module V"),
    ("What does DPO stand for?", ["direct", "preference", "optimization"], "Module V"),
    ("What does RLHF stand for?", ["reinforcement", "human", "feedback"], "Module V"),
    ("Which Python framework is used for the microservices in this course?", ["fastapi"], "Module VI"),
    ("Which tool is used for Infrastructure as Code on AWS in this course?", ["terraform"], "Module VI"),
    ("What does MCP stand for?", ["model", "context", "protocol"], "Module VI"),
]

# Convert questions into MongoDB documents
golden = [
    {
        "id": f"Q{i+1:02d}",
        "question": question,
        "expected_keywords": keywords,
        "source": source
    }
    for i, (question, keywords, source) in enumerate(GOLDEN)
]

# Save a local JSON backup
with open("golden_set.json", "w") as f:
    json.dump(golden, f, indent=2)

# Restore only if the collection is empty
if db.golden_set.count_documents({}) == 0:
    result = db.golden_set.insert_many(golden)
    print("Successfully inserted:", len(result.inserted_ids), "questions")
else:
    print("Golden set already contains:",
          db.golden_set.count_documents({}), "questions.")
    print("No existing data was overwritten.")

# Verify restoration
print("Total golden questions:",
      db.golden_set.count_documents({}))


Successfully inserted: 32 questions
Total golden questions: 32


In [19]:
for question in db.golden_set.find({}, {"_id": 0}).limit(5):
    print(question)
    print("-" * 50)

{'id': 'Q01', 'question': 'What does BPE stand for?', 'expected_keywords': ['byte', 'pair'], 'source': 'Module I'}
--------------------------------------------------
{'id': 'Q02', 'question': 'Which tokenizer does BERT use?', 'expected_keywords': ['wordpiece'], 'source': 'Module I'}
--------------------------------------------------
{'id': 'Q03', 'question': 'Which tokenizer family do T5 and mT5 use?', 'expected_keywords': ['sentencepiece'], 'source': 'Module I'}
--------------------------------------------------
{'id': 'Q04', 'question': 'Why does Telugu text usually cost more tokens than English?', 'expected_keywords': ['token'], 'source': 'Module I'}
--------------------------------------------------
{'id': 'Q05', 'question': 'What is a knowledge cutoff of an LLM?', 'expected_keywords': ['date'], 'source': 'Module I'}
--------------------------------------------------


In [21]:
# Verify Experiment 11 results in MongoDB

total = db.sampling_sweep.count_documents({})

print("Total saved experiment records:", total)

print("\nRecords per configuration:")

for config in db.sampling_sweep.distinct("config"):
    count = db.sampling_sweep.count_documents(
        {"config": config}
    )
    print(config, ":", count)

Total saved experiment records: 40

Records per configuration:
T=0.7, top_p=0.9 : 8
T=1.0, top_k=40 : 8
T=1.5, top_p=1.0 : 8
T=2.0, top_k=100 : 8
greedy T=0 : 8


In [23]:
print("Total records:",
      db.sampling_sweep.count_documents({}))

for config in db.sampling_sweep.distinct("config"):
    print(
        config,
        db.sampling_sweep.count_documents({"config": config})
    )

Total records: 40
T=0.7, top_p=0.9 8
T=1.0, top_k=40 8
T=1.5, top_p=1.0 8
T=2.0, top_k=100 8
greedy T=0 8
